# Traffic Congestion Predictor — Improved

Improvements over original:
- **BPR congestion model** instead of naive `flow / 100`
- **Richer features**: speed ratio, flow-to-capacity ratio, time-of-day bin, is_peak_hour flag
- **Realistic synthetic data**: 1000+ samples with peak/off-peak traffic patterns
- **Hyperparameter tuning** via RandomizedSearchCV
- **Richer evaluation**: MAE, RMSE, R², plus feature importance chart
- **Input validation and error handling**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
def bpr_congestion(flow, capacity, alpha=0.15, beta=4.0):
    """
    Bureau of Public Roads (BPR) congestion function.
    Returns a travel-time ratio: 1.0 = free flow, >1 = congested.
    Capped at 3.0 (extreme gridlock).

    Formula: t = t0 * (1 + alpha * (flow/capacity)^beta)
    We return the ratio (t / t0) normalised to [0, 1] for consistency
    with the original interface.
    """
    ratio = flow / max(capacity, 1)
    congestion_factor = 1.0 + alpha * (ratio ** beta)
    congestion_factor = min(congestion_factor, 3.0)          # cap
    # Normalise to roughly [0, 1]: free-flow = 0, gridlock = ~1
    return round((congestion_factor - 1.0) / 2.0, 4)

In [ ]:
class TrafficPredictor:
    """Random-Forest traffic congestion predictor with BPR ground truth."""

    PEAK_HOURS = {7, 8, 9, 17, 18, 19}   # AM and PM rush

    def __init__(self, capacity_default: int = 100):
        self.capacity_default = capacity_default
        self.model = None
        self.feature_names = [
            "edge_id",
            "hour_of_day",
            "time_period",
            "flow",
            "capacity",
            "flow_capacity_ratio",
            "normal_speed",
            "rush_speed",
            "speed_ratio",
            "is_peak_hour",
        ]

    # ------------------------------------------------------------------
    # Data loading
    # ------------------------------------------------------------------

    def load_data(self, traffic_store) -> pd.DataFrame:
        """
        Convert temporal traffic data into an ML-ready DataFrame.

        Expected traffic_store attributes
        ----------------------------------
        _flow_table  : dict[edge_key -> list[float]]   (flow per time period)
        _speed_table : dict[edge_key -> (normal_speed, rush_speed)]
        _capacity_table (optional) : dict[edge_key -> int]
        _hours_table    (optional) : dict[edge_key -> list[int]]  (hour-of-day per period)
        """
        rows = []

        for edge_key, flows in traffic_store._flow_table.items():
            normal_speed, rush_speed = traffic_store._speed_table.get(
                edge_key, (60, 40)
            )
            capacity = getattr(traffic_store, '_capacity_table', {}).get(
                edge_key, self.capacity_default
            )
            hours = getattr(traffic_store, '_hours_table', {}).get(
                edge_key, list(range(len(flows)))
            )

            edge_id = hash(edge_key) % 10_000
            speed_ratio = rush_speed / max(normal_speed, 1)

            for time_period, (flow, hour) in enumerate(zip(flows, hours)):
                fcr = flow / max(capacity, 1)
                congestion = bpr_congestion(flow, capacity)

                rows.append({
                    "edge_id":            edge_id,
                    "hour_of_day":        hour % 24,
                    "time_period":        time_period,
                    "flow":               flow,
                    "capacity":           capacity,
                    "flow_capacity_ratio": round(fcr, 4),
                    "normal_speed":       normal_speed,
                    "rush_speed":         rush_speed,
                    "speed_ratio":        round(speed_ratio, 4),
                    "is_peak_hour":       int(hour % 24 in self.PEAK_HOURS),
                    "congestion":         congestion,
                })

        df = pd.DataFrame(rows)
        print(f"Dataset: {len(df)} rows, {df['edge_id'].nunique()} unique edges")
        print(df.head())
        return df

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------

    def preprocess(self, df: pd.DataFrame):
        missing = [c for c in self.feature_names if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")

        X = df[self.feature_names]
        y = df["congestion"]
        return train_test_split(X, y, test_size=0.2, random_state=42)

    # ------------------------------------------------------------------
    # Training (with optional hyperparameter search)
    # ------------------------------------------------------------------

    def train(self, df: pd.DataFrame, tune: bool = True):
        X_train, X_test, y_train, y_test = self.preprocess(df)

        if tune:
            print("\nRunning RandomizedSearchCV (this may take a moment)...")
            param_dist = {
                "n_estimators":      [50, 100, 200, 300],
                "max_depth":         [None, 5, 10, 20],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf":  [1, 2, 4],
                "max_features":      ["sqrt", "log2", 0.8],
            }
            base = RandomForestRegressor(random_state=42, n_jobs=-1)
            search = RandomizedSearchCV(
                base, param_dist,
                n_iter=20, cv=5,
                scoring="neg_mean_absolute_error",
                random_state=42, n_jobs=-1
            )
            search.fit(X_train, y_train)
            self.model = search.best_estimator_
            print(f"Best params: {search.best_params_}")
        else:
            self.model = RandomForestRegressor(
                n_estimators=200, random_state=42, n_jobs=-1
            )
            self.model.fit(X_train, y_train)

        # 5-fold CV on training set
        cv_scores = cross_val_score(
            self.model, X_train, y_train,
            cv=5, scoring="r2"
        )
        print(f"\nCV R² (train): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        preds = self.model.predict(X_test)
        mae  = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds) ** 0.5
        r2   = r2_score(y_test, preds)

        print("\nTest-set Performance")
        print("--------------------")
        print(f"MAE   : {mae:.4f}")
        print(f"RMSE  : {rmse:.4f}")
        print(f"R²    : {r2:.4f}")

        self._print_feature_importance()
        return mae, rmse, r2

    # ------------------------------------------------------------------
    # Feature importance
    # ------------------------------------------------------------------

    def _print_feature_importance(self):
        if self.model is None:
            return
        importances = self.model.feature_importances_
        pairs = sorted(
            zip(self.feature_names, importances),
            key=lambda x: x[1], reverse=True
        )
        print("\nFeature Importances")
        print("-------------------")
        for feat, imp in pairs:
            bar = "█" * int(imp * 40)
            print(f"{feat:<22} {imp:.4f}  {bar}")

    # ------------------------------------------------------------------
    # Prediction
    # ------------------------------------------------------------------

    def predict_congestion(
        self,
        edge_id: int,
        hour_of_day: int,
        time_period: int,
        flow: float,
        capacity: float,
        normal_speed: float,
        rush_speed: float,
    ) -> float:
        if self.model is None:
            raise RuntimeError("Model not trained. Call .train() first.")
        if not (0 <= hour_of_day < 24):
            raise ValueError(f"hour_of_day must be 0-23, got {hour_of_day}")
        if flow < 0 or capacity <= 0:
            raise ValueError("flow must be ≥ 0 and capacity must be > 0")

        sample = pd.DataFrame([{
            "edge_id":             edge_id,
            "hour_of_day":         hour_of_day,
            "time_period":         time_period,
            "flow":                flow,
            "capacity":            capacity,
            "flow_capacity_ratio": flow / capacity,
            "normal_speed":        normal_speed,
            "rush_speed":          rush_speed,
            "speed_ratio":         rush_speed / max(normal_speed, 1),
            "is_peak_hour":        int(hour_of_day in self.PEAK_HOURS),
        }])
        return round(float(self.model.predict(sample)[0]), 4)

    @staticmethod
    def predict_traffic_level(congestion: float) -> str:
        """Map congestion score to a human-readable label."""
        if congestion < 0.10:
            return "Free Flow"
        elif congestion < 0.30:
            return "Low Traffic"
        elif congestion < 0.55:
            return "Medium Traffic"
        elif congestion < 0.75:
            return "High Traffic"
        else:
            return "Gridlock"

    # ------------------------------------------------------------------
    # Persistence
    # ------------------------------------------------------------------

    def save_model(self, path: str = "traffic_model.pkl"):
        if self.model is None:
            raise RuntimeError("Nothing to save — model not trained.")
        joblib.dump({"model": self.model, "features": self.feature_names}, path)
        print(f"Model saved to {path}")

    def load_model_file(self, path: str = "traffic_model.pkl"):
        payload = joblib.load(path)
        self.model = payload["model"]
        self.feature_names = payload["features"]
        print(f"Model loaded from {path}")

In [ ]:
# ------------------------------------------------------------------
# Realistic mock data generator
# ------------------------------------------------------------------

class MockTrafficStore:
    """
    Generates synthetic traffic data with realistic peak/off-peak patterns
    across N edges and 24 hourly time periods.
    """

    PEAK_HOURS = {7, 8, 9, 17, 18, 19}

    def __init__(self, n_edges: int = 50, seed: int = 42):
        rng = np.random.default_rng(seed)
        edges = [(chr(65 + i // 26) + chr(65 + i % 26), chr(65 + (i + 1) // 26) + chr(65 + (i + 1) % 26))
                 for i in range(n_edges)]

        self._flow_table     = {}
        self._speed_table    = {}
        self._capacity_table = {}
        self._hours_table    = {}

        for edge in edges:
            capacity      = int(rng.integers(60, 200))
            normal_speed  = int(rng.integers(50, 80))
            rush_speed    = int(rng.integers(20, normal_speed))

            hours = list(range(24))
            flows = []
            for h in hours:
                base = capacity * rng.uniform(0.1, 0.4)
                if h in self.PEAK_HOURS:
                    flow = base + capacity * rng.uniform(0.5, 1.1)   # near/over capacity
                elif h in {10, 11, 12, 13, 14, 15, 16}:
                    flow = base + capacity * rng.uniform(0.2, 0.5)
                else:
                    flow = base
                flows.append(round(max(flow, 0), 1))

            self._flow_table[edge]     = flows
            self._speed_table[edge]    = (normal_speed, rush_speed)
            self._capacity_table[edge] = capacity
            self._hours_table[edge]    = hours

In [ ]:
# ------------------------------------------------------------------
# Run
# ------------------------------------------------------------------

traffic_store = MockTrafficStore(n_edges=50)
predictor     = TrafficPredictor()

df = predictor.load_data(traffic_store)

# Use tune=False for a quick run; tune=True for best accuracy
mae, rmse, r2 = predictor.train(df, tune=False)

In [ ]:
# ------------------------------------------------------------------
# Example predictions
# ------------------------------------------------------------------

examples = [
    dict(label="Off-peak midnight",  hour_of_day=0,  flow=15,  capacity=100),
    dict(label="Morning rush",        hour_of_day=8,  flow=95,  capacity=100),
    dict(label="Evening rush (heavy)",hour_of_day=18, flow=130, capacity=100),
]

print(f"{'Scenario':<28} {'Congestion':>10}  Level")
print("-" * 55)
for ex in examples:
    cong = predictor.predict_congestion(
        edge_id=120,
        hour_of_day=ex["hour_of_day"],
        time_period=ex["hour_of_day"],
        flow=ex["flow"],
        capacity=ex["capacity"],
        normal_speed=60,
        rush_speed=35,
    )
    level = predictor.predict_traffic_level(cong)
    print(f"{ex['label']:<28} {cong:>10.4f}  {level}")

In [ ]:
# Save the model
predictor.save_model("traffic_model.pkl")